In [ ]:
import pandas as pd
import functions_simalign

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
df_1 = pd.read_csv("extraction_output/annotated_all_part_to_ger.csv").set_index("Unnamed: 0")
df_match_1 = pd.read_csv("extraction_output/adv_ger_matches.csv").set_index("Unnamed: 0")

In [ ]:
df_1 = pd.concat([df_1, df_match_1[["best_match", "best_score"]]], axis=1)

In [ ]:
df_1.loc[df_1["is_adv_participle"] & df_1["has_gerunziu"]].to_csv(
    "analysis/try_better_matching.csv"
)

In [ ]:
df_1 = df_1.loc[df_1["is_adv_participle"] & df_1["has_gerunziu"]]

In [ ]:
df_1.head(1)

In [ ]:
df_1["7"] = df_1["7"].str.replace("ş", "ș").replace("ţ", "ț")
df_1["gerunzius_and_lemmas"] = (
    df_1["gerunzius_and_lemmas"].str.replace("ş", "ș").replace("ţ", "ț") # TODO: this should happen at the very beginning
)

In [ ]:
for _, row in df_1.head(1).iterrows():
    print(row["3"])

In [ ]:
df_1

In [ ]:
matching_dict = {}
all_errors = []

In [ ]:
for i, row in df_1.iterrows():
    word_en = row["3"]
    sentence_en = row["english"]
    sentence_ro = row["7"]
    words_ro = [x[0] for x in eval(row["gerunzius_and_lemmas"])]
    truth_value, errors = functions_simalign.is_match(
        word_en=word_en,
        sentence_en=sentence_en,
        word_ro=words_ro,
        sentence_ro=sentence_ro,
    )
    if truth_value or (not truth_value and not errors):
        matching_dict[i] = truth_value
    if errors:
        for error in errors:
            print(f"{error.word=}, {error.sentence=}")
            all_errors.append(errors)

In [ ]:
len(errors)

In [ ]:
errors[25]

In [ ]:
functions_simalign.add_pronouns("întinzându", sentence='-Trebuie să-l pui și pe celălalt, a declarat el luând al doilea cercel și întinzându-mi-l.')

In [ ]:
len(matching_dict)

In [ ]:
matching_df = pd.DataFrame(
    zip(matching_dict.keys(), matching_dict.values()),
    columns=["index", "part_matches_gerunziu"],
).set_index("index")

In [ ]:
matching_df

In [ ]:
df_with_matching = pd.concat([df_1, matching_df], axis=1)

In [ ]:
df_with_matching["part_matches_gerunziu"].unique()

In [ ]:
df_with_matching.loc[
    ~(df_with_matching["part_matches_gerunziu"].isna())
    & (df_with_matching["part_matches_gerunziu"] == False)
].sample(10).to_csv("analysis/adv_part_wrong_gerunziu_simalign_sample.csv")

In [ ]:
sentence = "Vorbele ei îi înspăimântă grozav pe cei prezenţi: câteva păsări o tăiară la fugă, în graba mare; o Coţofană bătrână se înfoie în pene, zicând:-Trebuie să pornesc spre casă, nu-mi priește aerul nopţii la gât ! Un Canar își chemă puișorii, zicându-le:-Veniţi iute, dragii mei !"
word = "așteptându"

In [ ]:
functions_simalign.add_pronouns(word, sentence)

In [ ]:
import re

In [ ]:
re.split(';|,| ', "this, is a string to;, split")

In [ ]:
word = "hurrying"
sentence = "before her was another long passage, and the White Rabbit was still in sight, hurrying down it."

In [ ]:
splitting_chars = [" ", ",", ".", ":", ";"]
splitting_string = "|".join(splitting_chars)
words = [w for w in re.split(splitting_string, sentence) if w != ""]

In [ ]:
splitting_string

In [ ]:
re.split(';|,| |\\.', sentence)

In [ ]:
re.split(splitting_string, sentence)

In [ ]:
words

In [ ]:
re.match(f".*({word}-?[a-zăâîșț]*).*", sentence).group(1)

In [ ]:
test_sentence_en = "She return to the middle of the room, wondering how to get out of that place."
test_sentence_ro = "Se întoarse în mijlocul sălii, întrebându-și cum să facă să iasă din acel loc."

In [ ]:
import re

In [ ]:
word = "întrebându"

In [ ]:
matching = re.match(f".*({word}-[a-zăâîșț]*) .*", test_sentence_ro)

In [ ]:
matching.group(1)

In [ ]:
test_sentence_ro

In [ ]:
print(re.findall("cum", test_sentence_ro))

In [ ]:
functions_simalign.aligner.get_word_aligns(test_sentence_en, test_sentence_ro)

In [ ]:
df_1 = df_1[df_1["is_adv_participle"]] # only those are of our interest

In [ ]:
len(df_1.loc[df_1["has_gerunziu"] & (df_1["best_score"] > 0.117)])

In [ ]:
df_1.loc[~(df_1["has_gerunziu"] & (df_1["best_score"] > 0.117))].sample(10).to_csv(
    "analysis/adv_part_no_gerunziu_sample.csv"
)

In [ ]:
df_1.loc[(df_1["has_gerunziu"]) & ~(df_1["best_score"] > 0.117)].sample(20).to_csv(
    "analysis/adv_part_wrong_gerunziu_sample.csv"
)

In [ ]:
df_1.loc[df_1["has_gerunziu"] & (df_1["best_score"] > 0.12)].to_csv("analysis/part_to_gerunziu.csv")

In [ ]:
df_1.loc[~(df_1["has_gerunziu"] & df_1["best_score"] > 0.12)
].sample(20).to_csv("analysis/adv_part_no_gerunziu_sample.csv")

In [ ]:
df_2 = pd.read_csv("extraction_output/annotated_all_ger_to_part.csv").set_index("Unnamed: 0")
df_match_2 = pd.read_csv("extraction_output/ger_adv_matches.csv").set_index("Unnamed: 0")

In [ ]:
df_2 = pd.concat([df_2, df_match_2[["best_match", "best_score"]]], axis=1)

In [ ]:
df_2 = df_2.loc[df_2["is_gerunziu"]] # only those are of our interest

In [ ]:
len(df_2)

In [ ]:
df_2.loc[(df_2["has_adv_participle"])].head(10)

In [ ]:
len(df_2.loc[(df_2["has_adv_participle"]) & (df_2["best_score"] > 0.11)])

In [ ]:
df_2.loc[
    (df_2["has_adv_participle"])
    & (df_2["best_score"] > 0.11)
    & (df_2["best_score"] < 0.12)
].sample(10).to_csv("analysis/score_between_11_12_sample.csv")

In [ ]:
df_2.loc[(df_2["has_adv_participle"]) & (df_2["best_score"] > 0.117)].to_csv(
    "analysis/gerunziu_to_part.csv"
)

In [ ]:
df_ger.loc[(df["has_adv_participle"]) & ~(df_ger["best_score"] > 0.12)].to_csv("analysis/gerund_translated_to")